<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 4 · Curves, Swaps, and CIR Short Rates

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/04_dxquant_rates_curves_and_swaps.ipynb)


## Notebook Goals

This notebook streamlines the fixed-income building blocks in `dxquant`:

- build a single-curve zero curve by deposit/swap bootstrapping,
- value a plain fixed-for-floating interest rate swap on the curve, and
- use the CIR (1985) short-rate model for analytical bond pricing and calibration.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

ENABLE_PLOTS = False
plt = None
if ENABLE_PLOTS:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        plt = None

## 1. Bootstrapping a Zero Curve

We bootstrap a continuously compounded zero curve from synthetic deposit and par swap quotes.
The result is an `InterpolatedZeroCurve` that can provide discount factors, zero rates, and
forward rates.

In [ ]:
pricing_date = dt.date(2024, 1, 1)

def add_days(days: int) -> dt.date:
    return pricing_date + dt.timedelta(days=days)

deposits = [
    dx.DepositQuote(maturity=add_days(30), rate=0.0200),
    dx.DepositQuote(maturity=add_days(90), rate=0.0210),
]

swaps = [
    dx.SwapQuote(maturity=add_days(365), rate=0.0230, payments_per_year=1),
    dx.SwapQuote(maturity=add_days(730), rate=0.0250, payments_per_year=1),
    dx.SwapQuote(maturity=add_days(1095), rate=0.0265, payments_per_year=1),
]

curve = dx.bootstrap_zero_curve(
    pricing_date=pricing_date,
    deposits=deposits,
    swaps=swaps,
)

test_date = add_days(730)
print("discount factor:", curve.discount_factor(test_date))
print("zero rate:", curve.zero_rate(test_date))

## 2. Interest Rate Swap: Par Rate and Present Value

A par swap has present value zero by construction. Using a bootstrapped curve, the swap's
par rate is implied by the curve's discount factors.

In [ ]:
swap = dx.InterestRateSwap(
    name="irs",
    fixed_rate=0.0,
    notional=1_000_000.0,
    effective_date=pricing_date,
    termination_date=dt.date(2027, 1, 2),
    payment_frequency="6M",
)

par = swap.par_rate(curve)
print("par rate:", par)

par_swap = dx.InterestRateSwap(
    name="irs_par",
    fixed_rate=float(par),
    notional=1_000_000.0,
    effective_date=pricing_date,
    termination_date=dt.date(2027, 1, 2),
    payment_frequency="6M",
)

pv = par_swap.present_value(curve)
print("PV at par rate:", pv)

## 3. CIR Zero-Coupon Bond Pricing and Calibration

The CIR model provides closed-form zero-coupon bond prices. We generate synthetic target
prices from known parameters and recover them by least-squares calibration.

In [ ]:
initial_rate = 0.03
true = np.array([0.8, 0.05, 0.15])
maturities = np.array([0.5, 1.0, 2.0, 3.0])

targets = [
    dx.cir_zero_coupon(initial_rate, *true, float(t)) for t in maturities
]
fitted = dx.rates.cir.calibrate_cir(
    initial_rate=initial_rate,
    maturities=maturities,
    target_prices=targets,
    initial_guess=[0.5, 0.04, 0.1],
)

labels = ["kappa", "theta", "sigma"]
for label, value, target in zip(labels, fitted, true, strict=True):
    print(f"{label:>5s}: fitted={value: .6f}  target={target: .6f}")

## 4. CIR Path Simulation (Optional)

The Euler scheme produces short-rate paths. Plots are optional and disabled by default.

In [ ]:
process = dx.CIRShortRateProcess(
    kappa=float(true[0]),
    theta=float(true[1]),
    sigma=float(true[2]),
    seed=123,
)

r_paths = process.simulate_paths(
    short_rate=initial_rate,
    maturity=3.0,
    steps=120,
    paths=500,
)

mean_path = np.mean(r_paths, axis=0)
print("mean short rate at maturity:", float(mean_path[-1]))

if plt is not None:
    plt.figure(figsize=(10, 4))
    plt.plot(r_paths[:25].T, lw=1.0, alpha=0.7)
    plt.title("CIR short-rate sample paths")
    plt.xlabel("time step")
    plt.ylabel("short rate")
    plt.show()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">